# CudaRobotics — GPU MPPI & point-cloud registration, in your browser

[![GitHub](https://img.shields.io/badge/GitHub-CudaRobotics-181717?logo=github)](https://github.com/rsasaki0109/CudaRobotics)

This notebook builds the [`cudarobotics`](https://github.com/rsasaki0109/CudaRobotics/tree/v1.0.0/python) Python package from the immutable `v1.0.0` tag on a free Colab GPU and runs two demos:

1. **GPU MPPI planner** — thousands of sampled rollouts per control cycle (1 CUDA thread = 1 trajectory), steering a robot through a wall gap.
2. **GPU point-cloud registration** — FilterReg aligning a noisy, partially overlapping scan.

**Before running:** select a GPU runtime via *Runtime → Change runtime type → GPU*. The build takes ~3–5 minutes; everything after that runs in seconds.

In [ ]:
!nvidia-smi -L
!nvcc --version | tail -1

In [ ]:
import os
import subprocess

![ -d CudaRobotics ] || git clone --depth 1 --branch v1.0.0 https://github.com/rsasaki0109/CudaRobotics.git

# Compile only for this GPU's architecture (keeps the build short).
try:
    cap = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
    ).strip().splitlines()[0].replace(".", "")
    os.environ["CMAKE_ARGS"] = f"-DCMAKE_CUDA_ARCHITECTURES={cap}"
    print(f"building for sm_{cap}")
except Exception as exc:
    print("compute_cap query failed; using the default CUDA architecture:", exc)

%pip install ./CudaRobotics/python

import cudarobotics as cr
print("cudarobotics import OK")

## 1. GPU MPPI planner

A wall with a 2 m gap blocks the straight-line path. Every 50 ms control cycle the planner samples thousands of 56-step trajectories on the GPU, scores them against the costmap and reference path, and softmin-averages the best ones. This is the same CUDA core that powers the repo's [Nav2 controller plugin](https://github.com/rsasaki0109/CudaRobotics/tree/v1.0.0/ros2_ws/src/cuda_mppi_controller).

In [ ]:
import numpy as np

SIZE, RES = 200, 0.05  # 10 m x 10 m costmap
costmap = np.zeros((SIZE, SIZE), dtype=np.uint8)
wx0, wx1 = int(4.9 / RES), int(5.1 / RES)
gy0, gy1 = int(4.0 / RES), int(6.0 / RES)
costmap[:gy0, wx0:wx1] = 254  # wall below the gap
costmap[gy1:, wx0:wx1] = 254  # wall above the gap

path_x = np.arange(1.0, 9.05, 0.1, dtype=np.float32)
path = np.stack([path_x, np.full_like(path_x, 5.0)], axis=1)

K = 16384  # sampled rollouts per control cycle
planner = cr.MppiPlanner(batch_size=K, time_steps=56, model_dt=0.05)

state = np.array([1.0, 5.0, 0.0], dtype=np.float32)  # x, y, yaw
trajectory = [state[:2].copy()]
for _ in range(500):
    v, vy, w, info = planner.compute(
        state, costmap, path, (9.0, 5.0, 0.0), resolution=RES, goal_is_final=True
    )
    yaw = state[2]
    state[0] += 0.05 * (v * np.cos(yaw) - vy * np.sin(yaw))
    state[1] += 0.05 * (v * np.sin(yaw) + vy * np.cos(yaw))
    state[2] = np.arctan2(np.sin(yaw + 0.05 * w), np.cos(yaw + 0.05 * w))
    trajectory.append(state[:2].copy())
    if np.linalg.norm(state[:2] - np.array([9.0, 5.0])) < 0.25:
        break

print(f"goal reached in {len(trajectory) - 1} steps "
      f"({K} rollouts x 56 timesteps re-planned every cycle)")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import PillowWriter
from IPython.display import Image

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(costmap, origin="lower", extent=(0, 10, 0, 10), cmap="gray_r")
ax.plot([1, 9], [5, 5], "C0--", linewidth=1, label="reference path")
line, = ax.plot([], [], "C3", linewidth=2, label="MPPI trajectory")
point, = ax.plot([], [], "o", color="C3")
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect("equal")
ax.legend(loc="upper left")

writer = PillowWriter(fps=20)
with writer.saving(fig, "mppi_quickstart.gif", dpi=100):
    for i in range(2, len(trajectory) + 1, 2):
        xy = np.asarray(trajectory[:i])
        line.set_data(xy[:, 0], xy[:, 1])
        point.set_data([xy[-1, 0]], [xy[-1, 1]])
        writer.grab_frame()
plt.close(fig)
Image(filename="mppi_quickstart.gif")

### Solve time vs. sample count

The point of running MPPI on a GPU: scaling the rollout count barely costs wall-clock time. For reference, Nav2's stock CPU controller spends ~27 ms per solve at K=10,000 on a desktop i9 ([head-to-head benchmark](https://github.com/rsasaki0109/CudaRobotics/blob/v1.0.0/docs/results/cuda_mppi_vs_nav2_2026-06-10.md)).

In [ ]:
import time

bench_state = np.array([1.0, 5.0, 0.0], dtype=np.float32)
for k in (2048, 16384, 65536):
    p = cr.MppiPlanner(batch_size=k, time_steps=56, model_dt=0.05)
    for _ in range(3):  # warm-up
        p.compute(bench_state, costmap, path, (9.0, 5.0, 0.0),
                  resolution=RES, goal_is_final=True)
    t0 = time.perf_counter()
    n_iter = 20
    for _ in range(n_iter):
        p.compute(bench_state, costmap, path, (9.0, 5.0, 0.0),
                  resolution=RES, goal_is_final=True)
    dt_ms = (time.perf_counter() - t0) / n_iter * 1e3
    print(f"K={k:6d}: {dt_ms:6.2f} ms / solve")

## 2. GPU point-cloud registration (FilterReg)

Align a noisy, 85%-overlap copy of a lumpy closed surface back onto the original. The package also ships `Bcpd` (non-rigid), `SinkhornReg` (optimal transport), `Fgr`, `RobustTreg` (Student's-t), and `RobustP2Plane` under `cudarobotics.registration`.

In [ ]:
def make_lumpy(n, seed=1):
    rng = np.random.default_rng(seed)
    z = rng.uniform(-1.0, 1.0, n)
    phi = rng.uniform(0.0, 2.0 * np.pi, n)
    r2 = np.sqrt(np.clip(1.0 - z * z, 0.0, None))
    d = np.stack([r2 * np.cos(phi), r2 * np.sin(phi), z], axis=1)
    radius = (2.0 + 0.35 * np.sin(3.0 * phi) * (1.0 - z * z)
              + 0.30 * d[:, 2] * d[:, 0] + 0.20 * np.cos(2.0 * phi))
    bumps = [(0.8, 0.2, 0.5, 0.9, 0.25), (-0.3, 0.9, 0.2, 0.7, 0.30),
             (0.1, -0.6, 0.8, 0.8, 0.22), (-0.7, -0.4, -0.5, 1.0, 0.28),
             (0.4, 0.3, -0.85, 0.6, 0.20)]
    for bx, by, bz, height, width in bumps:
        ang = 1.0 - d @ np.array([bx, by, bz])
        radius += height * np.exp(-(ang * ang) / (2.0 * width * width))
    return (radius[:, None] * d).astype(np.float32)


def euler_to_rot(rx, ry, rz):
    c, s = np.cos, np.sin
    Rx = np.array([[1, 0, 0], [0, c(rx), -s(rx)], [0, s(rx), c(rx)]])
    Ry = np.array([[c(ry), 0, s(ry)], [0, 1, 0], [-s(ry), 0, c(ry)]])
    Rz = np.array([[c(rz), -s(rz), 0], [s(rz), c(rz), 0], [0, 0, 1]])
    return (Rx @ Ry @ Rz).astype(np.float32)


target = make_lumpy(8000, seed=1)
R_gt = euler_to_rot(0.12, -0.18, 0.08)
t_gt = np.array([0.35, -0.25, 0.20], dtype=np.float32)
rng = np.random.default_rng(7)
keep = rng.uniform(0.0, 1.0, len(target)) <= 0.85
source = ((target @ R_gt.T) + t_gt + rng.normal(0.0, 0.02, size=target.shape))
source = source[keep].astype(np.float32)

registrar = cr.registration.FilterReg()
t0 = time.perf_counter()
rotation, translation, info = registrar.register(target, source)
dt_ms = (time.perf_counter() - t0) * 1e3

R_est = np.array(rotation).reshape(3, 3)
aligned = source @ R_est.T + np.array(translation)
rot_err_deg = np.degrees(
    np.arccos(np.clip((np.trace(R_est @ R_gt) - 1.0) / 2.0, -1.0, 1.0))
)
print(f"{len(source)} -> {len(target)} points, {info['iterations']} EM iterations, "
      f"{dt_ms:.1f} ms")
print(f"rmse={info['final_rmse']:.4f}  residual rotation error={rot_err_deg:.2f} deg")

In [ ]:
sub = np.arange(0, len(source), max(1, len(source) // 1500))
fig, axes = plt.subplots(1, 2, figsize=(11, 5), subplot_kw={"projection": "3d"})
for ax, cloud, title in ((axes[0], source, "before"), (axes[1], aligned, "after")):
    ax.scatter(*target[sub].T, s=2, alpha=0.4, label="target")
    ax.scatter(*cloud[sub].T, s=2, alpha=0.4, label="source")
    ax.set_title(f"FilterReg: {title}")
    ax.set_box_aspect((1, 1, 1))
    ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## What else is in the box

- **Nav2 GPU MPPI controller plugin** — drop-in replacement for `nav2_mppi_controller`, 65k rollouts in ~10 ms: [`ros2_ws/src/cuda_mppi_controller/`](https://github.com/rsasaki0109/CudaRobotics/tree/v1.0.0/ros2_ws/src/cuda_mppi_controller)
- **~100 runnable CUDA demos** (SLAM, particle filters, planners, registration) with an [animated gallery](https://rsasaki0109.github.io/CudaRobotics/)
- **Benchmarks & reproductions** with fixed seeds and honest failure cases: [`docs/`](https://github.com/rsasaki0109/CudaRobotics/tree/v1.0.0/docs)

If this was useful, a ⭐ on [GitHub](https://github.com/rsasaki0109/CudaRobotics) helps the project.